In [1]:
#packages required
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import sent_tokenize
import nltk
import time

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

/opt/anaconda3/envs/nlp_environment/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists())
INPUT = project_root / 'data' / 'processed' / 'Executives_Earnings_Latest_PreEvent.csv'

Origional data check

In [6]:
# 1. Text field health
print("=== Text field health ===")
print(f"Null text:    {df['text'].isna().sum()}")
print(f"Empty string: {(df['text'].astype(str).str.strip() == '').sum()}")

# 2. Rows per company
rows_per_co = df.groupby('companyid').size()
print(f"\n=== Rows per company ===")
print(rows_per_co.describe())

# 3. Date sanity (pre-event check)
if 'call_date' in df.columns:
    print(f"\n=== Call date range ===")
    print(f"Min: {df['call_date'].min()}")
    print(f"Max: {df['call_date'].max()}  (should be < 2025-01-20)")

=== Text field health ===
Null text:    0
Empty string: 0

=== Rows per company ===
count    400.000000
mean      23.405000
std       13.798622
min        3.000000
25%       15.750000
50%       22.000000
75%       28.000000
max      177.000000
dtype: float64

=== Call date range ===
Min: 2024-10-02
Max: 2025-01-17  (should be < 2025-01-20)


Sentences prepared for finbert input

In [7]:
#split text into sentences 
def split_into_sentences(text):
    """Split a text block into sentences; return [] for null/empty input."""
    if pd.isna(text) or not str(text).strip():
        return []
    return sent_tokenize(str(text))

# Expand each row into one-sentence-per-row, keeping source metadata
print("Tokenizing sentences...")
t0 = time.time()

sentences_rows = []
for _, row in df.iterrows():
    sents = split_into_sentences(row['text'])
    for i, s in enumerate(sents):
        sentences_rows.append({
            'companyid':   row['companyid'],
            'companyname': row['companyname'],
            'row_idx':     row.name,
            'sent_idx':    i,
            'sentence':    s,
        })

sent_df = pd.DataFrame(sentences_rows)
print(f"Done in {time.time() - t0:.1f}s")
print(f"Total sentences: {len(sent_df):,}")
print(f"Avg sentences per row: {len(sent_df) / len(df):.1f}")

Tokenizing sentences...
Done in 0.9s
Total sentences: 120,942
Avg sentences per row: 12.9


Apply >= 3 sentence filter

In [ ]:

# Compute per-sentence word count
sent_df['word_count'] = sent_df['sentence'].str.split().str.len()

# Inspect distribution and threshold trade-offs (consistent with sample-run logic)
print("Word count distribution:")
print(sent_df['word_count'].describe())

print(f"\nTotal sentences: {len(sent_df):,}")
for threshold in [3, 5, 8, 10]:
    kept = (sent_df['word_count'] >= threshold).sum()
    print(f"  >={threshold} words: kept {kept:,} ({kept/len(sent_df)*100:.1f}%)")

# Apply >=3 words filter
sent_df_clean = sent_df[sent_df['word_count'] >= 3].copy().reset_index(drop=True)
print(f"\nAfter >=3 words filter: {len(sent_df_clean):,} sentences "
      f"({len(sent_df_clean)/len(sent_df)*100:.1f}% retained)")


Word count distribution:
count    120942.000000
mean         18.988416
std          12.051699
min           1.000000
25%          10.000000
50%          17.000000
75%          25.000000
max         170.000000
Name: word_count, dtype: float64

Total sentences: 120,942
  >=3 words: kept 115,525 (95.5%)
  >=5 words: kept 111,406 (92.1%)
  >=8 words: kept 101,762 (84.1%)
  >=10 words: kept 94,121 (77.8%)

After >=3 words filter: 115,525 sentences (95.5% retained)


In [ ]:
# sanity check:
coverage = sent_df_clean.groupby('companyid').size()
print(f"Companies with >=1 sentence after filter: {coverage.shape[0]} / 400")
print(f"\nSentences per company:")
print(coverage.describe())

# Check for companies dropped entirely (no sentence survived)
missing = 400 - coverage.shape[0]
if missing > 0:
    print(f"\nWARNING: {missing} companies lost all sentences (all too short)")
    missing_ids = set(df['companyid'].unique()) - set(coverage.index)
    print(f"Missing companyids: {list(missing_ids)[:10]}")
else:
    print("\nAll 400 companies retained")

# Companies with the fewest sentences (flag potential reliability issues)
print(f"\nCompanies with fewest sentences (bottom 5):")
bottom5 = coverage.sort_values().head(5)
for cid, n in bottom5.items():
    name = sent_df_clean[sent_df_clean['companyid'] == cid]['companyname'].iloc[0]
    print(f"  {cid} | {name}: {n} sentences")

Companies with >=1 sentence after filter: 400 / 400

Sentences per company:
count     400.000000
mean      288.812500
std       109.874421
min        55.000000
25%       216.000000
50%       279.500000
75%       350.000000
max      1370.000000
dtype: float64

All 400 companies retained

Companies with fewest sentences (bottom 5):
  247156 | Clearfield, Inc.: 55 sentences
  342660 | Socket Mobile, Inc.: 64 sentences
  691277895 | Motorsport Games Inc.: 74 sentences
  111756 | Sohu.com Limited: 94 sentences
  607620390 | Crown Electrokinetics Corp.: 98 sentences


FINBERT sentiment analysis 

In [11]:
MODEL_NAME = 'ProsusAI/finbert'

print("Loading FinBERT...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Select best available device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

model = model.to(device)
model.eval()  # inference mode (disables dropout, etc.)

print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Device: {device}")
print(f"Label mapping: {model.config.id2label}")

Loading FinBERT...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 50184.84it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded in 1.5s
Device: mps
Label mapping: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [13]:
def score_sentences_batch(sentences, batch_size=32, verbose=True):
    """
    Run FinBERT on a list of sentences.
    Returns a DataFrame with columns p_positive, p_negative, p_neutral.
    """
    # Preserve exact label ordering from the model config
    id2label = model.config.id2label
    label_order = [id2label[i] for i in sorted(id2label.keys())]

    results = []
    n = len(sentences)
    t0 = time.time()

    for i in range(0, n, batch_size):
        batch = sentences[i:i + batch_size]

        # Tokenize: pad to longest in batch, truncate to 512 tokens max
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        # Forward pass (no gradient needed for inference)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

        for p in probs:
            row = {f'p_{label_order[j]}': float(p[j]) for j in range(len(label_order))}
            results.append(row)

        # Progress logging every ~10k sentences
        if verbose and (i // batch_size) % 300 == 0 and i > 0:
            elapsed = time.time() - t0
            rate = (i + batch_size) / elapsed
            eta = (n - i) / rate
            print(f"  Progress: {i:,}/{n:,} | {rate:.0f} sent/sec | ETA {eta:.0f}s")

    elapsed = time.time() - t0
    if verbose:
        print(f"Scored {n:,} sentences in {elapsed:.1f}s ({n/elapsed:.1f} sent/sec)")
    return pd.DataFrame(results)

Sanity check: 3 sentences with clear positive / negative / neutral tone

In [ ]:
# # Sanity check: 3 sentences with clear positive / negative / neutral tone
# test_sents = [
#     "We achieved record revenue and our earnings exceeded all expectations.",
#     "We are facing significant losses and declining market share.",
#     "The company's fiscal year ends on December 31.",
# ]

# test_scores = score_sentences_batch(test_sents, batch_size=3, verbose=False)
# for sent, (_, row) in zip(test_sents, test_scores.iterrows()):
#     print(sent)
#     for col in test_scores.columns:
#         print(f"  {col}: {row[col]:.3f}")
#     print()

We achieved record revenue and our earnings exceeded all expectations.
  p_positive: 0.955
  p_negative: 0.016
  p_neutral: 0.028

We are facing significant losses and declining market share.
  p_positive: 0.008
  p_negative: 0.970
  p_neutral: 0.023

The company's fiscal year ends on December 31.
  p_positive: 0.017
  p_negative: 0.197
  p_neutral: 0.786



FINBERT for all filtered sentences

In [ ]:
# Run FinBERT on ALL filtered sentences (~83k)
sentences_to_score = sent_df_clean['sentence'].tolist()
print(f"Scoring {len(sentences_to_score):,} sentences...")

scores = score_sentences_batch(sentences_to_score, batch_size=32)

# Merge scores back to the sentence dataframe
sent_df_scored = pd.concat(
    [sent_df_clean.reset_index(drop=True), scores],
    axis=1
)

# Derived signed sentiment: positive minus negative
sent_df_scored['sentiment'] = sent_df_scored['p_positive'] - sent_df_scored['p_negative']

# Assign dominant label (argmax over three probabilities)
prob_cols = ['p_positive', 'p_negative', 'p_neutral']
sent_df_scored['dominant_label'] = (
    sent_df_scored[prob_cols].idxmax(axis=1).str.replace('p_', '')
)

print(f"\nScored shape: {sent_df_scored.shape}")
print(f"\nDominant label distribution:")
print(sent_df_scored['dominant_label'].value_counts())
print(f"\nDominant label proportion:")
print(sent_df_scored['dominant_label'].value_counts(normalize=True).round(3))

Scoring 115,525 sentences...
Estimated time: 7.7 minutes

  Progress: 9,600/115,525 | 327 sent/sec | ETA 324s
  Progress: 19,200/115,525 | 340 sent/sec | ETA 284s
  Progress: 28,800/115,525 | 341 sent/sec | ETA 255s
  Progress: 38,400/115,525 | 342 sent/sec | ETA 226s
  Progress: 48,000/115,525 | 346 sent/sec | ETA 195s
  Progress: 57,600/115,525 | 346 sent/sec | ETA 167s
  Progress: 67,200/115,525 | 348 sent/sec | ETA 139s
  Progress: 76,800/115,525 | 348 sent/sec | ETA 111s
  Progress: 86,400/115,525 | 349 sent/sec | ETA 83s
  Progress: 96,000/115,525 | 350 sent/sec | ETA 56s
  Progress: 105,600/115,525 | 351 sent/sec | ETA 28s
  Progress: 115,200/115,525 | 352 sent/sec | ETA 1s
Scored 115,525 sentences in 328.2s (352.0 sent/sec)

Scored shape: (115525, 11)

Dominant label distribution:
dominant_label
neutral     63942
positive    43073
negative     8510
Name: count, dtype: int64

Dominant label proportion:
dominant_label
neutral     0.553
positive    0.373
negative    0.074
Name: pr

In [17]:
# Firm-level aggregation of sentence-level scores
firm_features = sent_df_scored.groupby(['companyid', 'companyname']).agg(
    n_sentences=('sentence', 'count'),
    mean_p_positive=('p_positive', 'mean'),
    mean_p_negative=('p_negative', 'mean'),
    mean_p_neutral=('p_neutral', 'mean'),
    mean_sentiment=('sentiment', 'mean'),
    std_sentiment=('sentiment', 'std'),
).reset_index()

# Proportion of sentences in each dominant-label category
label_props = (
    sent_df_scored.groupby('companyid')['dominant_label']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .rename(columns=lambda x: f'pct_{x}')
    .reset_index()
)

# Ensure all three label columns exist even if some are missing
for col in ['pct_positive', 'pct_negative', 'pct_neutral']:
    if col not in label_props.columns:
        label_props[col] = 0.0

# Ratio of strongly negative sentences (p_negative > 0.5)
# This is the "ratio of negative statements" feature mentioned in the proposal
strong_neg = (
    sent_df_scored.assign(is_strong_neg=(sent_df_scored['p_negative'] > 0.5))
    .groupby('companyid')['is_strong_neg']
    .mean()
    .reset_index(name='ratio_strong_negative')
)

#Ratio of strongly positive sentences (p_positive > 0.5)
strong_pos = (
    sent_df_scored.assign(is_strong_pos=(sent_df_scored['p_positive'] > 0.5))
    .groupby('companyid')['is_strong_pos']
    .mean()
    .reset_index(name='ratio_strong_positive')
)


# Merge all firm-level features
firm_features = (
    firm_features
    .merge(label_props, on='companyid', how='left')
    .merge(strong_neg, on='companyid', how='left')
    .merge(strong_pos, on='companyid', how='left')
)

print(f"Firm-level features shape: {firm_features.shape}")
print(f"\nFirst 5 companies:")
print(firm_features.head())
print(f"\nFeature columns:")
print(firm_features.columns.tolist())

Firm-level features shape: (400, 13)

First 5 companies:
   companyid            companyname  n_sentences  mean_p_positive  mean_p_negative  mean_p_neutral  mean_sentiment  std_sentiment  pct_negative  pct_neutral  pct_positive  ratio_strong_negative  \
0      18749       Amazon.com, Inc.          308         0.424323         0.061255        0.514421        0.363068       0.429909      0.045455     0.558442      0.396104               0.035714   
1      19691    Cisco Systems, Inc.          429         0.414515         0.074649        0.510837        0.339866       0.460723      0.055944     0.543124      0.400932               0.051282   
2      21127      Intel Corporation          401         0.438403         0.096015        0.465582        0.342388       0.491449      0.079800     0.481297      0.438903               0.077307   
3      21171            Intuit Inc.          395         0.363213         0.068352        0.568435        0.294860       0.436720      0.055696     0.61265

Quick sanity check of firm-level features

In [18]:
# ============================================================
# EDA: Quick sanity check of firm-level features before export
# ============================================================

print("=" * 70)
print("1. Feature distribution summary")
print("=" * 70)
feature_cols = [c for c in firm_features.columns
                if c not in ['companyid', 'companyname']]
print(firm_features[feature_cols].describe().round(4))

print("\n" + "=" * 70)
print("2. Missing values check")
print("=" * 70)
null_counts = firm_features.isnull().sum()
if null_counts.sum() == 0:
    print("No missing values. All 400 firms have complete features.")
else:
    print(null_counts[null_counts > 0])

print("\n" + "=" * 70)
print("3. Top 10 most positive firms (by mean_sentiment)")
print("=" * 70)
top_pos = firm_features.nlargest(10, 'mean_sentiment')[
    ['companyname', 'n_sentences', 'mean_sentiment',
     'ratio_strong_positive', 'ratio_strong_negative']
]
print(top_pos.to_string(index=False))

print("\n" + "=" * 70)
print("4. Top 10 most negative firms (by mean_sentiment)")
print("=" * 70)
top_neg = firm_features.nsmallest(10, 'mean_sentiment')[
    ['companyname', 'n_sentences', 'mean_sentiment',
     'ratio_strong_positive', 'ratio_strong_negative']
]
print(top_neg.to_string(index=False))

print("\n" + "=" * 70)
print("5. Feature correlation matrix (should see intuitive signs)")
print("=" * 70)
corr_cols = ['mean_sentiment', 'std_sentiment',
             'ratio_strong_positive', 'ratio_strong_negative',
             'pct_positive', 'pct_negative']
print(firm_features[corr_cols].corr().round(3))

print("\n" + "=" * 70)
print("6. Outlier check: firms with extreme std_sentiment")
print("=" * 70)
std_threshold = firm_features['std_sentiment'].quantile(0.99)
outliers = firm_features[firm_features['std_sentiment'] > std_threshold][
    ['companyname', 'n_sentences', 'mean_sentiment', 'std_sentiment']
]
print(f"Firms above 99th percentile of std_sentiment ({std_threshold:.3f}):")
print(outliers.to_string(index=False))

1. Feature distribution summary
       n_sentences  mean_p_positive  mean_p_negative  mean_p_neutral  mean_sentiment  std_sentiment  pct_negative  pct_neutral  pct_positive  ratio_strong_negative  ratio_strong_positive
count     400.0000         400.0000         400.0000        400.0000        400.0000       400.0000      400.0000     400.0000      400.0000               400.0000               400.0000
mean      288.8125           0.3955           0.0963          0.5082          0.2991         0.4737        0.0788       0.5375        0.3837                 0.0755                 0.3793
std       109.8744           0.0760           0.0443          0.0792          0.0959         0.0650        0.0484       0.1001        0.0927                 0.0475                 0.0928
min        55.0000           0.1719           0.0286          0.2644         -0.0511         0.2987        0.0090       0.2467        0.1179                 0.0090                 0.1128
25%       216.0000           0.34

In [19]:
# File save
OUTPUT_FILENAME = 'firm_finbert_features.csv'
firm_features.to_csv(OUTPUT_FILENAME, index=False)

verify = pd.read_csv(OUTPUT_FILENAME)
print(f"Saved to: {OUTPUT_FILENAME}")
print(f"Shape on disk: {verify.shape}")
print(f"All columns match: {list(verify.columns) == list(firm_features.columns)}")
print(f"All firm IDs preserved: {set(verify['companyid']) == set(firm_features['companyid'])}")
print(f"\nFirst 3 rows of saved file:")
print(verify.head(3))

Saved to: firm_finbert_features.csv
Shape on disk: (400, 13)
All columns match: True
All firm IDs preserved: True

First 3 rows of saved file:
   companyid          companyname  n_sentences  mean_p_positive  mean_p_negative  mean_p_neutral  mean_sentiment  std_sentiment  pct_negative  pct_neutral  pct_positive  ratio_strong_negative  \
0      18749     Amazon.com, Inc.          308         0.424323         0.061255        0.514421        0.363068       0.429909      0.045455     0.558442      0.396104               0.035714   
1      19691  Cisco Systems, Inc.          429         0.414515         0.074649        0.510837        0.339866       0.460723      0.055944     0.543124      0.400932               0.051282   
2      21127    Intel Corporation          401         0.438403         0.096015        0.465582        0.342388       0.491449      0.079800     0.481297      0.438903               0.077307   

   ratio_strong_positive  
0               0.392857  
1               0.3939